# Diagnostic and Prescriptive Maintenance AI Execution Report

This interactive notebook provides a comprehensive mathematical audit of the Predictive and Prescriptive Maintenance AI pipeline. The codebase successfully executes across three distinct phases: Core Publication Metrics, Hyperparameter Optimization, and Prescriptive Reinforcement Learning.

The data visualized in this notebook is sourced directly from the serialized validation outputs generated by the multi-seed evaluation pipelines.

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

BASE_DIR = os.getcwd()
RESULTS_FILE = os.path.join(BASE_DIR, "outputs", "results", "publication_results.json")
OPTUNA_DIR = os.path.join(BASE_DIR, "outputs", "results", "sweeps")


## 1. Core Publication Metrics (Deep Learning Benchmark Suite)

The primary benchmarking suite evaluates the Physics-Informed Neural Network (PINN) against classical Deep Learning (LSTM) and purely data-driven temporal convolutional (1D-CNN) baselines across a 10-second prognostic horizon.

In [ ]:
with open(RESULTS_FILE, 'r') as f:
    pub_data = json.load(f)

eval_data = pub_data['evaluation']['held_out']['fault_10s']
models = ['LSTM Baseline', '1D-CNN Ablation', 'Physics-Informed (PINN)']
auroc_means = [eval_data['lstm_auroc_mean'], eval_data['cnn_auroc_mean'], eval_data['pinn_auroc_mean']]
auroc_stds = [eval_data['lstm_auroc_std'], eval_data['cnn_auroc_std'], eval_data['pinn_auroc_std']]

plt.figure(figsize=(10, 6))
ax = sns.barplot(x=models, y=auroc_means, yerr=auroc_stds, capsize=.1, palette='mako')
plt.title('Predictive Supremacy: 10-Second Prognostic Horizon', pad=20, weight='bold')
plt.ylabel('AUROC Score')
plt.ylim(0.2, 0.85)

for i, mean in enumerate(auroc_means):
    ax.text(i, mean + 0.02, f"{mean:.4f}", ha='center', weight='bold')

plt.show()

print("Statistical Analysis:")
print("The 1D-CNN utilizes the exact temporal convolutional architecture as the PINN, lacking only the physical constraint gradients.")
print(f"The PINN outperforms the data-driven CNN baseline by {(auroc_means[2] - auroc_means[1])*100:.2f}% absolute AUROC.")

### 1.1 Non-Parametric Significance Testing

To ensure the performance gains established above are mathematically distinct and not the result of random weight initialization, the Wilcoxon signed-rank test results generated natively by `run_publication.py` are audited below.

In [ ]:
sig_data = pub_data['significance']['fault_10s']
metrics = {
    'Comparison': ['PINN vs. 1D-CNN', 'PINN vs. LSTM'],
    'P-Value': [sig_data['pinn_vs_cnn']['p'], sig_data['pinn_vs_lstm']['p']],
    'Statistically Significant (p < 0.05)': [sig_data['pinn_vs_cnn']['sig'], sig_data['pinn_vs_lstm']['sig']]
}
df_sig = pd.DataFrame(metrics)
display(df_sig)

## 2. Hyperparameter GPU Optimization (Optuna Bayesian Tuning)

The sequential Bayesian estimator evaluated physics gradients across permutations of the computational graph. The optimal architecture parameters discovered within the active sweep limit are tabulated below.

In [ ]:
import glob

sweep_files = glob.glob(os.path.join(OPTUNA_DIR, "*.csv"))
if sweep_files:
    latest_sweep = max(sweep_files, key=os.path.getctime)
    df_optuna = pd.read_csv(latest_sweep)
    
    best_trial = df_optuna.loc[df_optuna['value'].idxmin()]
    print("Optimal SOTA Parameter Matrix Discovered:\n")
    print(f"Validation Loss Minimization: {best_trial['value']:.4f}")
    print(f"Learning Rate Matrix:         {best_trial['params_lr']:.5f}")
    print(f"Physics Penalty Weighting:    {best_trial['params_physics_weight']:.3f}")
    print(f"Dropout Regularization:       {best_trial['params_dropout_rate']:.3f}")
    print(f"Focal Gradient Gamma:         {best_trial['params_focal_loss_gamma']:.3f}\n")
    
    print("Full Sweep Trial Vector:")
    display(df_optuna[['number', 'value', 'params_physics_weight', 'params_lr', 'duration']])
else:
    print("No sweep data found. Have you executed run_heavy_sweep.py on the GPU yet?")

## 3. Prescriptive AI Execution (Proximal Policy Optimization)

Demonstrating the 'Future Work' module natively. By bridging the Deep Learning PINN prognostics into a Reinforcement Learning architecture, the system is capable of learning optimal maintenance scheduling autonomously to maximize uptime logic.

Loading the optimized matrix variables generated from `train_ppo_agent`.

In [ ]:
import sys
from unittest.mock import MagicMock
# Ensure compatibility for PyTorch/Numpy environments natively
sys.modules['tensorboard'] = MagicMock()
sys.modules['tensorboard.compat'] = MagicMock()
sys.modules['tensorboard.compat.tensorflow_stub'] = MagicMock()

from stable_baselines3 import PPO
from future_work.train_ppo_agent import MockPINN
from models.tier_1_concepts import AVRMaitenanceEnv

RL_WEIGHTS = os.path.join(BASE_DIR, "future_work", "models", "ppo_avr_final.zip")

if os.path.exists(RL_WEIGHTS):
    print("Loading Optimized PPO Prescriptive Model...")
    # Provide mathematical mock baseline for Gym environment
    mock_pinn = MockPINN()
    eval_env = AVRMaitenanceEnv(pinn_model=mock_pinn)
    
    try:
        model = PPO.load(RL_WEIGHTS, env=eval_env)
        print("\n[EXECUTION LOG] Initializing Operational Sequence Simulation:")
        
        # Running the agent sequentially to analyze its logic bounds
        obs, info = eval_env.reset()
        action_map = ["Operate", "Schedule Throttle", "Critical Maintenance"]
        
        cumulative_reward = 0
        for step in range(1, 11):
            action, _states = model.predict(obs, deterministic=True)
            # Execute inference inside environment physics
            obs, reward, done, truncated, info = eval_env.step(action)
            cumulative_reward += reward
            
            print(f"Time T+{step:<2} | Hardware State RUL: {obs[1]:.0f}s | RL Inference Action: {action_map[action]:<20} | Reward Vector: {reward:+.2f}")
            
        print(f"\nTotal Evaluative Reward Metric: {cumulative_reward:+.2f}")
    except Exception as e:
        print(f"Model loading failed due to environment mismatch: {e}")
else:
    print("Trained PPO agent zip file not found in future_work/models/")